#  TodoListMiddleware中间件
件赋予了Agent `任务规划` 和 `追踪进度` 的能力，可以 `应对复杂的多步任务` 。

`To-do list`的创建和维护是通过调用 `write_todos工具` 实现的

## 参数说明
### 参数1： system_prompt 自定义指导todo列表使用的提示词
不提供则使用内置提示词，通常不必提供。
### 参数2： tool_description —自定义write_tools工具的描述信息
不提供则使用内置描述，通常不必提供。

In [2]:
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen

custom_profile = {
"max_input_tokens": 128_000 # 最大上下文长度
}

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    profile=custom_profile, # 手动添加的配置项
)

In [5]:
from langchain.tools import tool
from pathlib import Path
import subprocess

WORKSPACE = Path("./todo_workspace")


@tool
def list_files(path: str = ".") -> str:
    """列出工作区指定目录下的文件和子目录。path 只能是相对路径。

    Args:
        path: 工作区下的相对路径，一定指向目录，默认为.，表示工作区根路径，不能访问工作区外的目录
    """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()
    if not str(target).startswith(str(workspace_root)):
        return "错误：只允许访问工作区内的目录。"
    if not target.exists():
        return f"错误：目录不存在: {path}"
    if not target.is_dir():
        return f"错误：不是目录: {path}"

    items = sorted(target.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    if not items:
        return f"目录为空: {path}"

    lines = []
    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")
    return "\n".join(lines)


@tool
def read_file(path: str) -> str:
    """读取工作区中的文本文件内容。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许读取工作区内的文件。"
    if not file_path.exists():
        return f"错误：文件不存在: {path}"
    return file_path.read_text(encoding="utf-8")


@tool
def write_file(path: str, content: str) -> str:
    """写入工作区中的文本文件。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
        content: 写入文件的内容
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许写入工作区内的文件。"
    file_path.write_text(content, encoding="utf-8")
    return f"已写入文件: {path}"


@tool
def run_tests() -> str:
    """在工作区运行 pytest -q，并返回输出。

    不接收任何参数，返回格式为
    returncode=0|1
    STDOUT:
    STDERR:
    """
    try:
        result = subprocess.run(
            ["pytest", "-q"],
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=20,
        )
        return (
            f"returncode={result.returncode}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )
    except Exception as e:
        return f"运行测试失败: {e}"


from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import HumanMessage
from rich import print as rprint

# 1. 初始化 Agent
agent = create_agent(
    model=model,
    # write_todos 等工具，TodoListMiddleware 需要配合这些工具使用
    tools=[list_files, read_file, write_file, run_tests],
    # 引入 Todo 列表中间件
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "你是一个代码修复助手。遇到多步骤任务时，先使用 write_todos 制定待办事项；"
        "然后读取文件、修复代码并运行测试。工作全部在工作区下进行。"
    ),
)

# 2. 使用 invoke 进行同步调用
print("正在执行 Agent 任务...")
final_state = agent.invoke(
    {
        "messages": [
            HumanMessage(content="请测试并修复工作区下 my_add.py 文件中的代码")
        ]
    }
)
rprint(final_state)

正在执行 Agent 任务...


{
    'messages': [
        HumanMessage(
            content='请测试并修复工作区下 my_add.py 文件中的代码',
            additional_kwargs={},
            response_metadata={},
            id='5dd6c267-a746-4822-97e1-9652d6424bd7'
        ),
        AIMessage(
            content='我来帮你测试并修复 `my_add.py` 文件。先让我读取文件内容并了解项目结构。\n\n',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '用户让我测试并修复工作区下 my_add.py 文件中的代码。我需要：\n1. 首先读取 
my_add.py 文件的内容\n2. 查看是否有相关的测试文件\n3. 运行测试看看有哪些问题\n4. 修复代码\n\n让我先读取文件内容。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 141,
                    'prompt_tokens': 1766,
                    'total_tokens': 1907,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 61,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 141
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 1766}
                },
                'model_provider': 'dashscope',
                'model_name': 'qwen3.6-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-3b8e635a-28ff-90d0-9fe2-d8243fd67101',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f276a-7006-7410-87db-dc4694b2a0a3-0',
            tool_calls=[
                {
                    'name': 'read_file',
                    'args': {'path': 'my_add.py'},
                    'id': 'call_41d31f2b2d864f0887df63fe',
                    'type': 'tool_call'
                },
                {
                    'name': 'list_files',
                    'args': {'path': '.'},
                    'id': 'call_0879406e9ee543b48cb3e099',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1766,
                'output_tokens': 141,
                'total_tokens': 1907,
                'input_token_details': {},
                'output_token_details': {'reasoning': 61}
            }
        ),
        ToolMessage(
            content='def add(a: int, b: int) -> int:\n    """返回两个整数的和"""\n    return a * b\n',
            name='read_file',
            id='92236420-e496-43c8-9764-ebe0ece00959',
            tool_call_id='call_41d31f2b2d864f0887df63fe'
        ),
        ToolMessage(
            content='[DIR] .pytest_cache\n[DIR] __pycache__\n[FILE] my_add.py\n[FILE] test_my_add.py',
            name='list_files',
            id='f2a5db96-6905-4c8e-ac60-af684738bd94',
            tool_call_id='call_0879406e9ee543b48cb3e099'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'refusal': None,
                'reasoning_content': '我发现了问题！`my_add.py` 中的 `add` 
函数应该返回两个整数的和，但实际实现使用的是乘法 (`a * b`) 而不是加法 (`a + 
b`)。让我先看看测试文件的内容，然后修复代码并运行测试。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 88,
                    'prompt_tokens': 1917,
                    'total_tokens': 2005,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 56,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 88
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 1917}
                },
                'model_provider': 'dashscope',
                'model_name': 'qwen3.6-flash',
                'system_fingerprint': None,
                'id